# 🚀 Master Class: Autonomous RAG (Retrieval-Augmented Generation)

## 📖 1. What is Autonomous RAG?

> **Autonomous RAG** is an advanced Retrieval-Augmented Generation paradigm where the AI agent operates **independently** with full self-management of **planning, retrieving, reflecting, correcting, and improving** — without requiring manual human steering at each step.

---

### 🌟 Key Pillars of Autonomous RAG:
1. **Agentic Reasoning & ReAct**: Dynamic reasoning loops (Reason $\rightarrow$ Act $\rightarrow$ Observe).
2. **Query Planning & Decomposition**: Splitting complex queries into structured sub-questions.
3. **Chain-of-Thought (CoT)**: Step-by-step intermediate reasoning paths.
4. **Multi-Source Retrieval**: Reading PDF documents, Vector DBs, Wikipedia, ArXiv, and external APIs.
5. **Iterative Retrieval & Refinement**: Detecting knowledge gaps and dynamically re-searching.
6. **Self-Reflection & Self-Correction**: Grading retrieval context and generated answers for hallucinations and completeness, triggering autonomous retry loops if needed.

---

### 📊 Comparison: Agentic RAG vs Autonomous RAG

| Feature | Agentic RAG | Autonomous RAG |
| :--- | :--- | :--- |
| **Definition** | RAG system using an agent where LLM reasons & uses tools | Fully self-managed system with planning, multi-source retrieval, reflection & self-healing |
| **Focus** | Structured tool usage (ReAct, LangGraph) | Complete autonomy in task execution, retries, and iterative refinement |
| **Behavior Flow** | `Think` $\rightarrow$ `Act` $\rightarrow$ `Observe` $\rightarrow$ `Answer` | `Plan` $\rightarrow$ `CoT` $\rightarrow$ `Act (ReAct)` $\rightarrow$ `Iterate` $\rightarrow$ `Synthesize` $\rightarrow$ `Reflect` $\rightarrow$ `Retry/Answer` |
| **Retry Logic** | Optional / static retry rules | Dynamic reflection-driven retry & critique loops |
| **Self-Reflection** | Optional add-on | **Core requirement**: reflects on context relevance & hallucination |
| **Planner** | Often manual or single-step | **Always present**: breaks complex queries into sub-questions |
| **Multi-Source** | Single tool or predefined sequence | Dynamic selection across PDFs, VectorStores, Wikipedia, ArXiv, etc. |

## 🏗️ 2. Complete Autonomous RAG Workflow Architecture (LangGraph)

```mermaid
flowchart TD
    Start([🚀 User Query]) --> Planner[1. Query Planning & Decomposition]
    Planner --> CoT[2. Chain of Thought Reasoning]
    CoT --> MultiRetrieve[3. Multi-Source ReAct Retrieval<br/>- Local PDF VectorStore<br/>- Wikipedia<br/>- ArXiv]
    MultiRetrieve --> IterativeCheck{4. Iterative Retrieval Check<br/>Sufficient Context?}
    IterativeCheck -- Missing Info / Refine --> MultiRetrieve
    IterativeCheck -- Sufficient Context --> Synthesizer[5. Multi-Source Answer Synthesis]
    Synthesizer --> Reflector{6. Self-Reflection & Verification<br/>Hallucination / Quality Check}
    Reflector -- Fail & Max Retries Not Reached --> MultiRetrieve
    Reflector -- Pass / Satisfied --> FinalAnswer([✅ Final Grounded Answer])
```

## 📦 Step 1: Install & Import Dependencies

In [ ]:
import os
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv

# LangChain Core & Models
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Embeddings & VectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Document Loaders & Tool Wrappers
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.document_loaders import ArxivLoader

# LangGraph State & Orchestration
from langgraph.graph import StateGraph, END

# Load environment variables (.env)
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")

## 🤖 Step 2: Initialize LLM & Embedding Models

In [ ]:
# Initialize Chat Model (supports GPT-4o, GPT-4o-mini, Groq, Ollama, etc.)
llm = init_chat_model("openai:gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings()
print("✅ LLM and Embeddings initialized successfully!")

## 📄 Step 3: Multi-Source Knowledge Base Setup (PDF, Vector DB, Wikipedia, ArXiv)

Autonomous RAG integrates diverse information sources:
- **Local PDF Knowledge Base**: Loads research papers (e.g., `attention.pdf` or custom docs) into FAISS Vector DB.
- **Wikipedia Tool**: Live encyclopedia search for broader context.
- **ArXiv Tool**: Scientific research paper search.

In [ ]:
# 1. Setup Local PDF / Document Retriever
pdf_path = "../0_DataInagestion/data/pdf/attention.pdf"

if os.path.exists(pdf_path):
    print(f"📄 Loading PDF from: {pdf_path}")
    pdf_loader = PyPDFLoader(pdf_path)
    raw_docs = pdf_loader.load()[:10]  # First 10 pages for quick indexing
else:
    print("ℹ️ Creating internal fallback technical documents...")
    raw_docs = [
        Document(
            page_content="The Transformer model relies entirely on self-attention mechanisms to compute representations of its input and output without using sequence-aligned RNNs or convolution. Multi-Head Attention allows the model to jointly attend to information from different representation subspaces at different positions. Scaled Dot-Product Attention computes attention scores using Query, Key, and Value matrices.",
            metadata={"source": "attention_paper.pdf", "page": 1}
        ),
        Document(
            page_content="Positional Encodings are added to the input embeddings at the bottoms of the encoder and decoder stacks to inject sequence order information. The positional encodings have the same dimension d_model as the embeddings, using sine and cosine functions of different frequencies.",
            metadata={"source": "attention_paper.pdf", "page": 2}
        )
    ]

# Split into chunks and create VectorStore
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(raw_docs)
vectorstore = FAISS.from_documents(chunks, embeddings)
pdf_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 2. Setup Wikipedia Tool
wiki_wrapper = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=1000)
wikipedia_tool = WikipediaQueryRun(api_wrapper=wiki_wrapper)

# 3. Setup ArXiv Search Helper Function
def search_arxiv(query: str, max_results: int = 2) -> str:
    try:
        docs = ArxivLoader(query=query, load_max_docs=max_results).load()
        if not docs:
            return "No ArXiv papers found for query."
        return "\n\n".join([f"[ArXiv: {d.metadata.get('Title', 'Paper')}]\n{d.page_content[:600]}" for d in docs])
    except Exception as e:
        return f"ArXiv search unavailable: {str(e)}"

print("✅ Multi-Source Retrievers (PDF FAISS, Wikipedia, ArXiv) ready!")

## 📊 Step 4: Define Autonomous RAG Agent State

The state maintains the complete execution trace, including:
- `query`: Original user query.
- `sub_queries`: Decomposed sub-questions from query planning.
- `cot_steps`: Chain-of-thought reasoning plan.
- `retrieved_docs`: Aggregated documents with source metadata.
- `draft_answer`: Synthesized intermediate response.
- `reflection`: Self-critique, hallucination check, and verification status.
- `retry_count` & `max_retries`: Safe iterative retry management.

In [ ]:
class AutonomousRAGState(BaseModel):
    query: str
    sub_queries: List[str] = Field(default_factory=list)
    cot_reasoning: str = ""
    retrieved_contexts: Dict[str, str] = Field(default_factory=dict)
    draft_answer: str = ""
    is_grounded: bool = False
    reflection_critique: str = ""
    retry_count: int = 0
    max_retries: int = 2
    final_answer: str = ""

## 🧩 Step 5: Implement Core Autonomous Agent Nodes

### 1. `query_planner_node`: Query Planning & Decomposition
Decomposes complex requests into distinct, targeted sub-queries.

In [ ]:
def query_planner_node(state: AutonomousRAGState) -> Dict[str, Any]:
    print(f"\n--- [1] 🎯 QUERY PLANNING & DECOMPOSITION ---")
    prompt = f"""You are an expert Autonomous RAG Planner.
Given the user query, decompose it into 2-3 specific, logical sub-questions to retrieve relevant information from local documents (PDFs), Wikipedia, and scientific archives (ArXiv).

User Query: {state.query}

Output each sub-question on a new line prefixed with '- '.
Do not add introductory remarks."""
    
    response = llm.invoke(prompt)
    sub_qs = [line.strip("- ").strip() for line in response.content.split("\n") if line.strip()]
    print(f"📌 Decomposed into {len(sub_qs)} sub-queries:")
    for sq in sub_qs:
        print(f"   • {sq}")
        
    return {"sub_queries": sub_qs}

### 2. `chain_of_thought_node`: Chain-of-Thought (CoT) Reasoning
Formulates a step-by-step reasoning roadmap for how to investigate and synthesize the information.

In [ ]:
def chain_of_thought_node(state: AutonomousRAGState) -> Dict[str, Any]:
    print(f"\n--- [2] 🧠 CHAIN OF THOUGHT (CoT) REASONING ---")
    prompt = f"""You are an autonomous AI reasoning engine.
Original Query: {state.query}
Sub-Questions to investigate: {state.sub_queries}

Explain your step-by-step reasoning roadmap (Chain-of-Thought) for solving this query accurately using multi-source retrieval (PDFs, Wikipedia, ArXiv) and synthesis."""
    
    response = llm.invoke(prompt)
    print(f"💡 CoT Roadmap formulated (length: {len(response.content)} chars)")
    return {"cot_reasoning": response.content}

### 3. `multi_source_retrieval_node`: ReAct Multi-Source Retrieval
Dynamically retrieves context from Local PDF VectorStore, Wikipedia, and ArXiv.

In [ ]:
def multi_source_retrieval_node(state: AutonomousRAGState) -> Dict[str, Any]:
    print(f"\n--- [3] 🔍 RE-ACT MULTI-SOURCE RETRIEVAL (Attempt {state.retry_count + 1}) ---")
    retrieved = dict(state.retrieved_contexts)
    
    # 1. Retrieve from Local PDF VectorStore
    for sq in state.sub_queries:
        pdf_docs = pdf_retriever.invoke(sq)
        if pdf_docs:
            combined_pdf = "\n".join([d.page_content for d in pdf_docs])
            retrieved[f"PDF (Query: {sq})"] = combined_pdf
            print(f"   📚 Retrieved from PDF for: '{sq}'")
    
    # 2. Retrieve from Wikipedia
    try:
        wiki_res = wikipedia_tool.run(state.query)
        retrieved["Wikipedia"] = wiki_res
        print(f"   🌐 Retrieved from Wikipedia for: '{state.query}'")
    except Exception as e:
        print(f"   ⚠️ Wikipedia search skipped: {e}")
        
    # 3. Retrieve from ArXiv
    arxiv_res = search_arxiv(state.query, max_results=1)
    if arxiv_res:
        retrieved["ArXiv"] = arxiv_res
        print(f"   🔬 Retrieved from ArXiv for: '{state.query}'")
        
    return {"retrieved_contexts": retrieved}

### 4. `iterative_check_node`: Iterative Retrieval Evaluation
Checks if retrieved documents adequately cover all sub-questions or if query terms require refinement.

In [ ]:
def iterative_check_node(state: AutonomousRAGState) -> Dict[str, Any]:
    print(f"\n--- [4] 🔁 ITERATIVE RETRIEVAL CHECK ---")
    total_sources = len(state.retrieved_contexts)
    print(f"📊 Total source contexts collected: {total_sources}")
    
    # Evaluates context volume and coverage
    if total_sources == 0 and state.retry_count < state.max_retries:
        print("⚠️ No context retrieved. Refining sub-queries...")
        refined_sub_qs = [f"{sq} machine learning fundamentals" for sq in state.sub_queries]
        return {"sub_queries": refined_sub_qs, "retry_count": state.retry_count + 1}
        
    return {}

### 5. `answer_synthesis_node`: Answer Synthesis
Fuses information from all sources into a well-structured, comprehensive answer.

In [ ]:
def answer_synthesis_node(state: AutonomousRAGState) -> Dict[str, Any]:
    print(f"\n--- [5] ✍️ MULTI-SOURCE ANSWER SYNTHESIS ---")
    
    formatted_context = ""
    for src, content in state.retrieved_contexts.items():
        formatted_context += f"\n### Source: {src}\n{content}\n"
        
    prompt = f"""You are an Autonomous RAG Synthesizer.
Answer the original query thoroughly based strictly on the provided multi-source context.

Original Query: {state.query}
Chain of Thought Guide: {state.cot_reasoning}

Multi-Source Context:
{formatted_context}

Instructions:
- Synthesize a cohesive, comprehensive, and professional explanation.
- Explicitly cite sources (e.g. [PDF], [Wikipedia], [ArXiv]) where relevant.
- Structure with clear headings and bullet points."""

    response = llm.invoke(prompt)
    print(f"📝 Draft Answer synthesized ({len(response.content)} chars)")
    return {"draft_answer": response.content}

### 6. `self_reflection_node`: Self-Reflection & Verification
Autonomously evaluates the draft answer for:
- Groundedness (no hallucinations)
- Query completeness
- Factual consistency

In [ ]:
def self_reflection_node(state: AutonomousRAGState) -> Dict[str, Any]:
    print(f"\n--- [6] 🪞 SELF-REFLECTION & HALLUCINATION CHECK ---")
    
    prompt = f"""You are an autonomous RAG quality evaluator and verifier.
Original Query: {state.query}
Draft Answer: {state.draft_answer}

Perform a critical evaluation:
1. Is the answer accurate and grounded in technical facts?
2. Does it directly address the user query without hallucinating unmentioned facts?
3. Is the formatting clear and helpful?

Output format:
STATUS: [PASS or RETRY]
CRITIQUE: [Provide 1-2 sentences of feedback or praise]"""

    response = llm.invoke(prompt)
    critique_text = response.content
    is_pass = "STATUS: PASS" in critique_text.upper() or "PASS" in critique_text.upper()
    
    print(f"🔍 Reflection Status: {'✅ PASS' if is_pass else '❌ RETRY'}")
    print(f"💬 Critique: {critique_text}")
    
    if is_pass or state.retry_count >= state.max_retries:
        return {
            "is_grounded": True,
            "reflection_critique": critique_text,
            "final_answer": state.draft_answer
        }
    else:
        return {
            "is_grounded": False,
            "reflection_critique": critique_text,
            "retry_count": state.retry_count + 1
        }

## 🔄 Step 6: Define Conditional Edge & Build LangGraph Workflow

In [ ]:
def reflection_router(state: AutonomousRAGState) -> str:
    """Decides whether to finalize the answer or perform an autonomous retry."""
    if state.is_grounded or state.retry_count >= state.max_retries:
        return "end"
    else:
        print(f"🔁 Reflection failed. Triggering autonomous retry loop (Attempt {state.retry_count + 1})...")
        return "retry"

# Build StateGraph
workflow = StateGraph(AutonomousRAGState)

# Add Nodes
workflow.add_node("planner", query_planner_node)
workflow.add_node("chain_of_thought", chain_of_thought_node)
workflow.add_node("multi_retrieve", multi_source_retrieval_node)
workflow.add_node("iterative_check", iterative_check_node)
workflow.add_node("synthesizer", answer_synthesis_node)
workflow.add_node("reflector", self_reflection_node)

# Define Flow / Edges
workflow.set_entry_point("planner")
workflow.add_edge("planner", "chain_of_thought")
workflow.add_edge("chain_of_thought", "multi_retrieve")
workflow.add_edge("multi_retrieve", "iterative_check")
workflow.add_edge("iterative_check", "synthesizer")
workflow.add_edge("synthesizer", "reflector")

# Conditional Edge from Reflector
workflow.add_conditional_edges(
    "reflector",
    reflection_router,
    {
        "end": END,
        "retry": "multi_retrieve"
    }
)

# Compile the Autonomous RAG Graph
app = workflow.compile()
print("🎉 Autonomous RAG LangGraph Workflow Compiled Successfully!")

## 📈 Step 7: Visualize Workflow Graph

In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Visual rendering note: {e}")

## 🧪 Step 8: Execute Autonomous RAG on Real-World Complex Query

In [ ]:
test_query = "How does the Transformer self-attention mechanism work, why are positional encodings necessary, and what makes it superior to traditional RNN architectures?"

initial_state = AutonomousRAGState(
    query=test_query,
    max_retries=2
)

print(f"🚀 STARTING AUTONOMOUS RAG WORKFLOW FOR QUERY:")
print(f"❓ '{test_query}'\n" + "="*70)

final_output = app.invoke(initial_state)

## 🎯 Step 9: View the Grounded Final Answer

In [ ]:
from IPython.display import Markdown, display

print("=" * 70)
print("🏁 AUTONOMOUS RAG FINAL SYNTHESIZED & VERIFIED ANSWER:")
print("=" * 70)
display(Markdown(final_output["final_answer"]))

## 📋 Summary & Key Takeaways

1. **Autonomous Planning**: The system breaks unstructured, open-ended questions into modular sub-tasks without requiring prompt engineering per question.
2. **Multi-Source Hybrid Retrieval**: Pulls from local indexed PDFs, online encyclopedias (Wikipedia), and scientific research repositories (ArXiv).
3. **Self-Healing & Reflection**: Actively critiques its own answers and re-executes search/synthesis if facts are missing or ungrounded.
4. **Production-Ready LangGraph Architecture**: Built with modern typed states, robust conditional branching, and clean cyclical workflows.